# Вспомогательные данные, часть 1: физическое кондиционирование планетой

> **Post-fix, 29.07.2026.** Source-код ниже исправлен: planet classifier fit
> выполняется только на `known ∩ train`. Сохранённые outputs этого notebook
> относятся к старому leaky run и не являются актуальными. Clean-метрики:
> `CLEAN_RUN_REPORT.md`, артефакты: `runs/aux_conditioning_train_only`.

Датасет содержит планету только как метку `PName`. Но за меткой стоят реальные
физические величины — масса и радиус планеты, орбита, температура, параметры
звезды. Они определяют глубину потенциальной ямы, скорость убегания газа и
масштаб профиля поглощения.

**Идея:** скачать эти параметры из каталога
[NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/)
(файл `aux_data/planet_params.csv`) и подать в модель **вектор физических
параметров системы** вместо дискретной метки планеты.

Чтобы не было утечки (на инференсе планета неизвестна), используем **мягкое
самокондиционирование**: сеть сама предсказывает планету по профилю
(отдельная голова), а softmax-вероятности взвешивают три физических вектора.
Полученная смесь подаётся на вход регрессионных голов. Внешняя информация
на инференсе не нужна — только сам профиль.

База — пайплайн из `dl_pro.ipynb` (WTA-мультиголова × 5 сетей + ранкер +
отбор 5 разнообразных кандидатов); сравниваем с его результатами
(best-of-5: XUV 10.8%, He 9.7%, logMsw 0.6%).

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier
from scipy.spatial.distance import cdist

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

device: mps


## 1. Данные и физические векторы планет

In [2]:
DATA_DIR = "new_dataset_V3"
GRID = np.arange(-5.0, 5.0 + 1e-9, 0.1)

profiles, targets, pnames = [], [], []
for name in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, name)
    if not os.path.isdir(path):
        continue
    params = {}
    with open(os.path.join(path, "parameters.txt")) as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 2:
                params[parts[0]] = parts[1]
    arr = np.loadtxt(os.path.join(path, "Absorption.dat"), skiprows=1)
    if arr[-1, 0] - arr[0, 0] < 5:
        continue
    profiles.append(np.interp(GRID, arr[:, 0], arr[:, 1], left=0.0, right=0.0))
    targets.append([float(params["XUVInt"]), float(params["Helium"]),
                    float(params["Msw"]), int(params["H2a"])])
    pnames.append(params.get("PName", "<нет>"))

profiles = np.array(profiles, dtype=np.float32)
targets = np.array(targets)
pnames = np.array(pnames)
y_reg = np.log10(targets[:, :3])
y_cls = targets[:, 3].astype(np.float32)
idx_train, idx_val = train_test_split(
    np.arange(len(profiles)), test_size=0.2, random_state=SEED, stratify=y_cls)
y_mean, y_std = y_reg[idx_train].mean(0), y_reg[idx_train].std(0)
x_scale = profiles[idx_train].max()

# PCA-16 (нужен для псевдометок и ранкера)
log_prof = np.log10(np.clip(profiles, 1e-7, None))
mu = log_prof[idx_train].mean(0)
_, S, Vt = np.linalg.svd(log_prof[idx_train] - mu, full_matrices=False)
Z = (log_prof - mu) @ Vt[:16].T
Z = ((Z - Z[idx_train].mean(0)) / Z[idx_train].std(0)).astype(np.float32)
print("профилей:", len(profiles), " train:", len(idx_train), " val:", len(idx_val))

профилей: 613  train: 490  val: 123


In [3]:
# 31 прогон без корректной метки планеты -> псевдометки классификатором по профилю
known = np.isin(pnames, ["WASP107b", "WASP69b", "WASP52b"])
# Важно: auxiliary classifier fit только на train; validation не влияет на train-псевдометки.
fit_idx = idx_train[known[idx_train]]
clf = GradientBoostingClassifier(random_state=0).fit(Z[fit_idx], pnames[fit_idx])
labels = pnames.copy()
labels[~known] = clf.predict(Z[~known])
PLANETS = {"WASP107b": 0, "WASP69b": 1, "WASP52b": 2}
planet = np.array([PLANETS[l] for l in labels])
print("псевдометки для", (~known).sum(), "прогонов:",
      dict(zip(*np.unique(labels[~known], return_counts=True))))

псевдометки для 31 прогонов: {'WASP107b': 20, 'WASP52b': 4, 'WASP69b': 7}


In [4]:
# физические векторы планет из каталога NASA Exoplanet Archive
cat = pd.read_csv("aux_data/planet_params.csv")
cat["pl_name"] = cat["pl_name"].str.replace(" ", "").str.replace("-", "")
COLS = ["pl_bmassj", "pl_radj", "pl_orbsmax", "pl_orbper", "pl_eqt",
        "pl_dens", "st_teff", "st_rad", "st_mass"]
PHYS = np.stack([np.log10(cat.loc[cat.pl_name == k, COLS].values[0].astype(float))
                 for k in PLANETS])
PHYS = (PHYS - PHYS.mean(0)) / (PHYS.std(0) + 1e-9)     # (3, 9)
PHYS_t = torch.tensor(PHYS, dtype=torch.float32).to(DEVICE)
pd.DataFrame(PHYS, index=list(PLANETS), columns=COLS).round(2)

,pl_bmassj,pl_radj,pl_orbsmax,pl_orbper,pl_eqt,pl_dens,st_teff,st_rad,st_mass
WASP107b,-1.32,-0.93,1.01,1.07,-1.21,-1.41,-1.33,-1.41,-1.39
WASP69b,0.22,-0.46,0.35,0.27,-0.04,0.71,0.24,0.78,0.47
WASP52b,1.10,1.39,-1.36,-1.34,1.24,0.70,1.08,0.63,0.92


## 2. Модель: WTA-мультиголова + мягкое кондиционирование

Отличия от `dl_pro`:
* добавлена **голова планеты** (3 класса, cross-entropy, вес 0.3);
* регрессионные головы получают на вход `[признаки энкодера ⊕ физвектор]`,
  где физвектор = softmax(планета) @ PHYS — дифференцируемая смесь
  каталожных параметров трёх планет.

In [5]:
def make_input(profile):
    linear = profile / x_scale
    log = np.log10(np.clip(profile, 1e-7, None))
    log = (log - log.mean()) / (log.std() + 1e-9)
    return np.stack([linear, log]).astype(np.float32)


class ProfileDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices = indices
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        profile = profiles[idx]
        if self.augment:
            profile = np.clip(profile + np.random.normal(
                0, 0.005 * profile.max(), profile.shape), 1e-7, None)
        return (torch.tensor(make_input(profile)),
                torch.tensor((y_reg[idx] - y_mean) / y_std, dtype=torch.float32),
                torch.tensor(y_cls[idx]),
                torch.tensor(planet[idx]))


val_loader = DataLoader(ProfileDataset(idx_val), batch_size=256)
N_HEADS = 5


class CondCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(2, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Flatten())
        flat = 128 * 12
        self.head_planet = nn.Sequential(nn.Linear(flat, 64), nn.ReLU(), nn.Linear(64, 3))
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(flat + 9, 256), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(256, 3)) for _ in range(N_HEADS)])
        self.head_cls = nn.Sequential(
            nn.Linear(flat, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))

    def forward(self, x):
        h = self.encoder(x)
        pl_logit = self.head_planet(h)
        phys = torch.softmax(pl_logit, -1) @ PHYS_t        # мягкий физвектор
        h_cond = torch.cat([h, phys], -1)
        reg = torch.stack([head(h_cond) for head in self.heads], 1)
        return reg, self.head_cls(h).squeeze(-1), pl_logit

## 3. Обучение ансамбля (5 сетей × 5 голов)

In [6]:
bce, ce = nn.BCEWithLogitsLoss(), nn.CrossEntropyLoss()
true_log = y_reg[idx_val]
true_lin = 10 ** true_log


def mape(t, p):
    return 100 * np.mean(np.abs((p - t) / t))


def train_model(seed, epochs=400):
    torch.manual_seed(seed)
    np.random.seed(seed)
    tl = DataLoader(ProfileDataset(idx_train, augment=True),
                    batch_size=64, shuffle=True)
    m = CondCNN().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best, best_state = float("inf"), None
    for ep in range(epochs):
        m.train()
        for x, y, c, p in tl:
            x, y, c, p = x.to(DEVICE), y.to(DEVICE), c.to(DEVICE), p.to(DEVICE)
            reg, cls, pl = m(x)
            per = ((reg - y.unsqueeze(1)) ** 2).mean(-1)
            loss = (per.min(1).values.mean() + 0.1 * per.mean()
                    + bce(cls, c) + 0.3 * ce(pl, p))
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
        m.eval()
        v = 0.0
        with torch.no_grad():
            for x, y, c, p in val_loader:
                x, y, c, p = x.to(DEVICE), y.to(DEVICE), c.to(DEVICE), p.to(DEVICE)
                reg, cls, pl = m(x)
                per = ((reg - y.unsqueeze(1)) ** 2).mean(-1)
                v += (per.min(1).values.mean() + 0.1 * per.mean()
                      + bce(cls, c) + 0.3 * ce(pl, p)).item() * len(x)
        v /= len(idx_val)
        if v < best:
            best = v
            best_state = {k: t.cpu().clone() for k, t in m.state_dict().items()}
    m.load_state_dict(best_state)
    m.eval()
    PR, PC, PL_ = [], [], []
    with torch.no_grad():
        for x, _, _, _ in val_loader:
            reg, cls, pl = m(x.to(DEVICE))
            PR.append(reg.cpu().numpy())
            PC.append(torch.sigmoid(cls).cpu().numpy())
            PL_.append(pl.argmax(-1).cpu().numpy())
    return (np.concatenate(PR) * y_std + y_mean,
            np.concatenate(PC), np.concatenate(PL_))


val_cands, val_proba = [], []
for seed in range(5):
    vp, vc, vpl = train_model(seed)
    val_cands.append(vp)
    val_proba.append(vc)
    o_x = 100 * np.mean(np.abs(10**vp[:, :, 0] - true_lin[:, 0:1]).min(1) / true_lin[:, 0])
    o_h = 100 * np.mean(np.abs(10**vp[:, :, 1] - true_lin[:, 1:2]).min(1) / true_lin[:, 1])
    print(f"модель {seed}: oracle-of-5 XUV {o_x:.1f}%  He {o_h:.1f}%  "
          f"AUC {roc_auc_score(y_cls[idx_val], vc):.3f}  "
          f"планета acc {(vpl == planet[idx_val]).mean():.3f}")

val_cands = np.concatenate([v[:, None, :, :] for v in val_cands], 1)
val_cands = val_cands.reshape(len(idx_val), -1, 3)          # (N, 25, 3)
val_proba = np.mean(val_proba, 0)

модель 0: oracle-of-5 XUV 15.8%  He 11.4%  AUC 0.994  планета acc 0.943


модель 1: oracle-of-5 XUV 10.9%  He 10.3%  AUC 0.997  планета acc 0.959


модель 2: oracle-of-5 XUV 12.4%  He 11.3%  AUC 0.996  планета acc 0.919


модель 3: oracle-of-5 XUV 12.7%  He 9.7%  AUC 0.998  планета acc 0.951


модель 4: oracle-of-5 XUV 11.9%  He 10.1%  AUC 0.999  планета acc 0.959


## 4. Ранкер и отбор 5 разнообразных кандидатов

(идентично `dl_pro.ipynb`)

In [7]:
torch.manual_seed(0)
np.random.seed(0)
yn = (y_reg - y_mean) / y_std
Ztr, yn_tr = Z[idx_train], yn[idx_train]
D_prof, D_par = cdist(Ztr, Ztr), cdist(yn_tr, yn_tr)
pair_x, pair_y = [], []
for a in range(len(idx_train)):
    pair_x.append(np.concatenate([Ztr[a], yn_tr[a]])); pair_y.append(1.0)
    for _ in range(2):
        pair_x.append(np.concatenate([Ztr[a], yn_tr[a] + np.random.normal(0, 0.05, 3)]))
        pair_y.append(1.0)
    for _ in range(3):
        b = np.random.randint(len(idx_train))
        if D_par[a, b] > 0.5:
            pair_x.append(np.concatenate([Ztr[a], yn_tr[b]])); pair_y.append(0.0)
    for b in [b for b in np.argsort(D_prof[a])[1:15] if D_par[a, b] > 0.8][:3]:
        pair_x.append(np.concatenate([Ztr[a], yn_tr[b]])); pair_y.append(0.0)
pair_x = torch.tensor(np.array(pair_x), dtype=torch.float32)
pair_y = torch.tensor(np.array(pair_y), dtype=torch.float32)

ranker = nn.Sequential(nn.Linear(19, 256), nn.ReLU(), nn.Dropout(0.2),
                       nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.2),
                       nn.Linear(256, 1))
opt = torch.optim.Adam(ranker.parameters(), lr=1e-3, weight_decay=1e-5)
for _ in range(3000):
    idx = torch.randperm(len(pair_x))[:256]
    loss = bce(ranker(pair_x[idx]).squeeze(-1), pair_y[idx])
    opt.zero_grad()
    loss.backward()
    opt.step()
ranker.eval()

cands_norm = (val_cands - y_mean) / y_std
feat = np.concatenate([np.repeat(Z[idx_val][:, None, :], 25, 1), cands_norm], -1)
with torch.no_grad():
    scores = ranker(torch.tensor(feat, dtype=torch.float32)
                    .reshape(-1, 19)).reshape(len(idx_val), 25).numpy()


def select_diverse(c, s, k=5):
    order = np.argsort(-s)
    chosen = [order[0]]
    for _ in range(k - 1):
        best_j, best_g = None, -np.inf
        for j in order:
            if j in chosen:
                continue
            g = min(np.linalg.norm(c[j] - c[i]) for i in chosen) + 0.02 * s[j]
            if g > best_g:
                best_g, best_j = g, j
        chosen.append(best_j)
    return chosen


top5 = np.stack([val_cands[i][select_diverse(cands_norm[i], scores[i])]
                 for i in range(len(idx_val))])

## 5. Результаты

In [8]:
def oracle5(j, log_scale=False):
    if log_scale:
        return 100 * np.mean(np.abs(top5[:, :, j] - true_log[:, j:j+1]).min(1)
                             / np.abs(true_log[:, j]))
    return 100 * np.mean(np.abs(10**top5[:, :, j] - true_lin[:, j:j+1]).min(1)
                         / true_lin[:, j])


auc = roc_auc_score(y_cls[idx_val], val_proba)
print("параметр | + физ. кондиционирование | dl_pro (без) | публикация")
print("---------+--------------------------+--------------+-----------")
print(f"XUVInt   | {oracle5(0):8.1f}%                | 10.8%        | 20.4%")
print(f"Helium   | {oracle5(1):8.1f}%                | 9.7%         | 17.9%")
print(f"logMsw   | {oracle5(2, True):8.1f}%                | 0.6%         | 1.8%")
print(f"H2a AUC  | {auc:8.3f}                 | 0.995        | 0.989")

параметр | + физ. кондиционирование | dl_pro (без) | публикация
---------+--------------------------+--------------+-----------
XUVInt   |      9.8%                | 10.8%        | 20.4%
Helium   |      8.9%                | 9.7%         | 17.9%
logMsw   |      0.5%                | 0.6%         | 1.8%
H2a AUC  |    0.998                 | 0.995        | 0.989


## Выводы

* Физическое кондиционирование даёт стабильное, хотя и умеренное улучшение
  всех метрик (~10% относительных к сильному пайплайну).
* На текущей валидации планеты те же, что в обучении, поэтому физвектор
  работает как «умный one-hot». Главный потенциал схемы — **обобщение на
  новые планеты**: сеть связывает форму профиля с непрерывными физическими
  величинами, а не с тремя метками. Это раскрывается в связке с генерацией
  синтетических профилей для других планет (см. `aux_pretrain.ipynb`).